# Gold Layer Table Schemas (Databricks)

## Gold Layer Table: `03_gold.d_sales_summary`

### Schema

| Column             | Data Type           | Description                |
|--------------------|---------------------|----------------------------|
| order_date         | DATE (Primary Key)  | Date of the order          |
| total_orders       | BIGINT              | Total number of orders     |
| total_revenue      | DECIMAL(12,2)       | Total revenue              |
| avg_order_value    | DECIMAL(10,2)       | Average order value        |
| unique_customers   | BIGINT              | Unique customers           |
| unique_restaurants | BIGINT              | Unique restaurants         |
| dine_in_orders     | BIGINT              | Dine-in orders             |
| takeaway_orders    | BIGINT              | Takeaway orders            |
| delivery_orders    | BIGINT              | Delivery orders            |

---

**SQL DDL:**
```
sql
CREATE TABLE `03_gold`.d_sales_summary (
    order_date DATE PRIMARY KEY,
    total_orders BIGINT,
    total_revenue DECIMAL(12,2),
    avg_order_value DECIMAL(10,2),
    unique_customers BIGINT,
    unique_restaurants BIGINT,
    dine_in_orders BIGINT,
    takeaway_orders BIGINT,
    delivery_orders BIGINT
)
```

In [0]:
df_orders=spark.read.table("databricksproject_ws.`02_silver`.fact_orders")
display(df_orders.limit(5))

In [0]:
'''
order_date DATE PRIMARY KEY,
total_orders BIGINT,
total_revenue DECIMAL(12,2),
avg_order_value DECIMAL(10,2),
unique_customers BIGINT,
unique_restaurants BIGINT,
dine_in_orders BIGINT,
takeaway_orders BIGINT,
delivery_orders BIGINT
 '''
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

df_sales = (
    df_orders
        .groupBy("order_date")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            sum("total_amount").cast(DecimalType(12, 2)).alias("total_revenue"),
            avg("total_amount").cast(DecimalType(10, 2)).alias("avg_order_value"),
            countDistinct("customer_id").alias("unique_customers"),
            countDistinct("restaurant_id").alias("unique_restaurants"),
            sum(when(col("order_type") == "dine_in", 1).otherwise(0)).alias("dine_in_orders"),
            sum(when(col("order_type") == "takeaway", 1).otherwise(0)).alias("takeaway_orders"),
            sum(when(col("order_type") == "delivery", 1).otherwise(0)).alias("delivery_orders")
        )
)

df_sales = df_sales.select(
    "order_date",
    "total_orders",
    "total_revenue",
    "avg_order_value",
    "unique_customers",
    "unique_restaurants",
    "dine_in_orders",
    "takeaway_orders",
    "delivery_orders"
)

display(df_sales)

In [0]:
from pyspark import pipelines as dp
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime


@dp.materialized_view(
    name="03_gold.d_sales_summary",
    partition_cols=["order_date"],
    table_properties={"quality": "gold"},
    comment="Gold layer aggregates with date-based overwrites",
)
def d_sales_summary():
    df_daily_agg = (
        dp.read("02_silver.fact_orders")
        .groupBy("order_date")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum("total_amount").cast("decimal(10,2)").alias("total_revenue"),
            F.avg("total_amount").cast("decimal(10,2)").alias("avg_order_value"),
            F.countDistinct("customer_id").alias("unique_customers"),
            F.countDistinct("restaurant_id").alias("unique_restaurants"),
            F.sum(F.when(F.col("order_type") == "dine_in", 1).otherwise(0)).alias(
                "dine_in_orders"
            ),
            F.sum(F.when(F.col("order_type") == "takeaway", 1).otherwise(0)).alias(
                "takeaway_orders"
            ),
            F.sum(F.when(F.col("order_type") == "delivery", 1).otherwise(0)).alias(
                "delivery_orders"
            ),
        )
        .select(
            "order_date",
            "total_orders",
            "total_revenue",
            "avg_order_value",
            "unique_customers",
            "unique_restaurants",
            "dine_in_orders",
            "takeaway_orders",
            "delivery_orders",
        )
    )
    return df_daily_agg
